In [ ]:
import os

!git clone https://github.com/sp-uhh/sgmse.git
os.chdir('/kaggle/working/sgmse')

!pip install -r requirements.txt --quiet
!pip install pesq pystoi pandas gdown --quiet

In [ ]:
!pip install numpy==1.26.4 --force-reinstall --quiet
# restart the session after this

In [ ]:
import os
os.chdir('/kaggle/working/sgmse')
import numpy as np
print(np.__version__)  # should be 1.26.4

In [ ]:
!gdown 1_H3EXvhcYBhOZ9QNUcD5VZHc6ktrRbwQ -O voicebank_pretrained.ckpt

In [ ]:
patch = """
import torch
_original_torch_load = torch.load
def _patched_torch_load(*args, **kwargs):
    kwargs['weights_only'] = False
    return _original_torch_load(*args, **kwargs)
torch.load = _patched_torch_load
"""

with open('/kaggle/working/sgmse/enhancement.py', 'r') as f:
    content = f.read()

content = content.replace('import torch\n', 'import torch\n' + patch, 1)

with open('/kaggle/working/sgmse/enhancement.py', 'w') as f:
    f.write(content)

print("enhancement.py patched")

In [ ]:
# APPLYING PHYSICS LOSS

physics_loss_fn = '''
    def _wave_equation_loss(self, x_td):
        d1 = x_td[:, 1:] - x_td[:, :-1]
        d2 = d1[:, 1:] - d1[:, :-1]
        return torch.mean(d2 ** 2)

'''

old_loss_line = "                loss = losses_tf + self.l1_weight * losses_l1\n"
new_loss_line = """                # physics loss
                losses_wave = self._wave_equation_loss(x_hat_td)
                loss = losses_tf + self.l1_weight * losses_l1 + self.physics_weight * losses_wave\n"""

old_init_line = "        self.loss_type = loss_type\n"
new_init_line = "        self.loss_type = loss_type\n        self.physics_weight = 0.1\n"

with open('/kaggle/working/sgmse/sgmse/model.py', 'r') as f:
    content = f.read()

content = content.replace('    def _loss(self', physics_loss_fn + '    def _loss(self', 1)
content = content.replace(old_init_line, new_init_line, 1)
content = content.replace(old_loss_line, new_loss_line, 1)

with open('/kaggle/working/sgmse/sgmse/model.py', 'w') as f:
    f.write(content)

# verify
!grep -n "physics\|wave_equation" /kaggle/working/sgmse/sgmse/model.py

In [ ]:
!pip install numpy==1.26.4 pandas==2.2.2 pyarrow==15.0.2 "datasets==2.20.0" --force-reinstall --quiet

In [4]:
import soundfile as sf
import numpy as np
from datasets import load_dataset, Audio

TEST_DIR = "data/test"
TRAIN_DIR = "data/train"
os.makedirs(f"{TEST_DIR}/clean", exist_ok=True)
os.makedirs(f"{TEST_DIR}/noisy", exist_ok=True)
os.makedirs(f"{TRAIN_DIR}/clean", exist_ok=True)
os.makedirs(f"{TRAIN_DIR}/noisy", exist_ok=True)

print("Loading test set...")
test_data = load_dataset("MeiWu1123/VoiceBank-DEMAND-16k", split="test")
test_data = test_data.cast_column("clean", Audio(sampling_rate=16000))
test_data = test_data.cast_column("noisy", Audio(sampling_rate=16000))

for i, sample in enumerate(test_data):
    clean = np.array(sample["clean"]["array"], dtype=np.float32)
    noisy = np.array(sample["noisy"]["array"], dtype=np.float32)
    fname = f"{i:04d}.wav"
    sf.write(f"{TEST_DIR}/clean/{fname}", clean, 16000)
    sf.write(f"{TEST_DIR}/noisy/{fname}", noisy, 16000)
print(f"Wrote {len(test_data)} test samples")

print("Loading train set (500 samples)...")
train_data = load_dataset("MeiWu1123/VoiceBank-DEMAND-16k", split="train")
train_data = train_data.cast_column("clean", Audio(sampling_rate=16000))
train_data = train_data.cast_column("noisy", Audio(sampling_rate=16000))

for i in range(1250):
    sample = train_data[i]
    clean = np.array(sample["clean"]["array"], dtype=np.float32)
    noisy = np.array(sample["noisy"]["array"], dtype=np.float32)
    fname = f"{i:04d}.wav"
    sf.write(f"{TRAIN_DIR}/clean/{fname}", clean, 16000)
    sf.write(f"{TRAIN_DIR}/noisy/{fname}", noisy, 16000)
print("Wrote 1250 train samples")

VALID_DIR = "data/valid"
os.makedirs(f"{VALID_DIR}/clean", exist_ok=True)
os.makedirs(f"{VALID_DIR}/noisy", exist_ok=True)

print("Writing validation set (100 samples)...")
for i in range(1250, 1350):
    sample = train_data[i]
    clean = np.array(sample["clean"]["array"], dtype=np.float32)
    noisy = np.array(sample["noisy"]["array"], dtype=np.float32)
    fname = f"{i:04d}.wav"
    sf.write(f"{VALID_DIR}/clean/{fname}", clean, 16000)
    sf.write(f"{VALID_DIR}/noisy/{fname}", noisy, 16000)
print("Wrote 100 validation samples")

Loading test set...


Generating train split:   0%|          | 0/11572 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/824 [00:00<?, ? examples/s]

Wrote 824 test samples
Loading train set (500 samples)...
Wrote 1250 train samples
Writing validation set (100 samples)...
Wrote 100 validation samples


In [ ]:
finetune_script = '''
import os
os.chdir('/kaggle/working/sgmse')
import torch
_original_torch_load = torch.load
def _patched_torch_load(*args, **kwargs):
    kwargs['weights_only'] = False
    return _original_torch_load(*args, **kwargs)
torch.load = _patched_torch_load
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, Callback
from sgmse.model import ScoreModel
from sgmse.data_module import SpecsDataModule

CKPT_PATH = "/kaggle/working/sgmse/voicebank_pretrained.ckpt"
SAVE_DIR = "/kaggle/working/sgmse_physics_finetuned"
DATA_DIR = "/kaggle/working/sgmse/data"
EPOCHS = 10
LR = 1e-5
os.makedirs(SAVE_DIR, exist_ok=True)

model = ScoreModel.load_from_checkpoint(
    CKPT_PATH,
    map_location="cuda",
    loss_type="data_prediction",
    loss_weighting="1",
    l1_weight=0.001,
    pesq_weight=0.0,
    num_eval_files=0,
    lr=LR
)
model.physics_weight = 0.001

class PrintValLoss(Callback):
    def on_validation_epoch_end(self, trainer, pl_module):
        metrics = trainer.callback_metrics
        val_loss = metrics.get("valid_loss")
        train_loss = metrics.get("train_loss_epoch")
        if val_loss is not None and train_loss is not None:
            print("\\n[Epoch " + str(trainer.current_epoch) + "] train_loss=" + str(round(float(train_loss), 4)) + " | valid_loss=" + str(round(float(val_loss), 4)) + "\\n")
        elif val_loss is not None:
            print("\\n[Epoch " + str(trainer.current_epoch) + "] valid_loss=" + str(round(float(val_loss), 4)) + "\\n")

data_module = SpecsDataModule(
    base_dir=DATA_DIR,
    format="default",
    batch_size=4,
    n_fft=510,
    hop_length=128,
    num_frames=256,
    window="hann",
    num_workers=2,
    dummy=False,
    spec_factor=0.15,
    spec_abs_exponent=0.5,
    normalize="noisy",
    transform_type="exponent"
)

checkpoint_callback = ModelCheckpoint(
    dirpath=SAVE_DIR,
    filename="physics_epoch{epoch:02d}_valloss{valid_loss:.4f}",
    save_top_k=3,
    monitor="valid_loss",
    mode="min",
    every_n_epochs=1
)

trainer = pl.Trainer(
    max_epochs=EPOCHS,
    accelerator="gpu",
    devices=1,
    callbacks=[checkpoint_callback, PrintValLoss()],
    log_every_n_steps=10,
    enable_progress_bar=True
)

trainer.fit(model, datamodule=data_module)
print("Best checkpoint: " + checkpoint_callback.best_model_path)
'''

with open('/kaggle/working/sgmse/finetune_physics.py', 'w') as f:
    f.write(finetune_script)

print("Script written successfully")

In [ ]:
with open('/kaggle/working/sgmse/sgmse/model.py', 'r') as f:
    lines = f.readlines()

# find and remove the duplicate and replace with fixed version
new_lines = []
skip_next = 0
i = 0
while i < len(lines):
    if '_wave_equation_loss' in lines[i] and 'def ' in lines[i]:
        # skip this function definition (4 lines: def, d1, d2, return)
        if skip_next == 0:
            # first occurrence - skip it entirely
            skip_next = 1
            i += 5  # skip def + d1 + d2 + return + blank line
            continue
        else:
            # second occurrence - replace with fixed version
            new_lines.append('    def _wave_equation_loss(self, x_td):\n')
            new_lines.append('        if x_td.dim() == 1:\n')
            new_lines.append('            x_td = x_td.unsqueeze(0)\n')
            new_lines.append('        d1 = x_td[:, 1:] - x_td[:, :-1]\n')
            new_lines.append('        d2 = d1[:, 1:] - d1[:, :-1]\n')
            new_lines.append('        return torch.mean(d2 ** 2)\n')
            i += 5
            continue
    new_lines.append(lines[i])
    i += 1

with open('/kaggle/working/sgmse/sgmse/model.py', 'w') as f:
    f.writelines(new_lines)

# verify
!grep -n -A 7 "_wave_equation_loss" /kaggle/working/sgmse/sgmse/model.py

In [ ]:
!python finetune_physics.py

In [ ]:
!sed -i 's/weights_only: Optional\[bool\] = None,/weights_only: Optional[bool] = False,/' /usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py

!sed -i 's/weights_only: Optional\[bool\] = None,/weights_only: Optional[bool] = False,/' /usr/local/lib/python3.12/dist-packages/lightning_fabric/utilities/cloud_io.py

# verify both patches
!grep -n "weights_only" /usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py
!grep -n "weights_only" /usr/local/lib/python3.12/dist-packages/lightning_fabric/utilities/cloud_io.py

In [ ]:
with open('sanity_check.py', 'w') as f:
    f.write("""
import os
os.chdir('/kaggle/working/sgmse')

# patch torch.load before anything else imports it
import torch
_original_torch_load = torch.load
def _patched_torch_load(*args, **kwargs):
    kwargs['weights_only'] = False
    return _original_torch_load(*args, **kwargs)
torch.load = _patched_torch_load

from sgmse.model import ScoreModel

model = ScoreModel.load_from_checkpoint(
    'voicebank_pretrained.ckpt',
    map_location='cpu',
    loss_type='data_prediction',
    loss_weighting='1',
    l1_weight=0.001,
    pesq_weight=0.0,
    num_eval_files=0,
    lr=1e-5
)
print('model loaded ok')
print('physics_weight:', model.physics_weight)
""")

!python sanity_check.py

In [ ]:
with open('/kaggle/working/sgmse/enhancement.py', 'r') as f:
    content = f.read()

# Remove all the duplicate patch blocks, keep only one
import re
patch_block = """import torch
_original_torch_load = torch.load
def _patched_torch_load(*args, **kwargs):
    kwargs['weights_only'] = False
    return _original_torch_load(*args, **kwargs)
torch.load = _patched_torch_load"""

# Remove all occurrences
while patch_block in content:
    content = content.replace(patch_block, '')

# Add a single clean patch at the top
clean_patch = """import glob
import torch

_original_torch_load = torch.load
def _patched_torch_load(*args, **kwargs):
    kwargs['weights_only'] = False
    return _original_torch_load(*args, **kwargs)
torch.load = _patched_torch_load

"""

# Remove any stray duplicate 'import glob' and 'import torch' at top
content = content.lstrip()
content = re.sub(r'^(import glob\n|import torch\n)+', '', content)
content = clean_patch + content

with open('/kaggle/working/sgmse/enhancement.py', 'w') as f:
    f.write(content)

print("Fixed. New top of file:")
with open('/kaggle/working/sgmse/enhancement.py', 'r') as f:
    print(f.read()[:400])

In [ ]:
CKPT = "/kaggle/working/sgmse_physics_finetuned/physics_epochepoch=09_vallossvalid_loss=25.3103.ckpt"

!python enhancement.py \
    --test_dir data/test/noisy \
    --enhanced_dir enhanced_physics \
    --ckpt $CKPT \
    --N 10

In [ ]:
!python calc_metrics.py \
    --clean_dir data/test/clean \
    --noisy_dir data/test/noisy \
    --enhanced_dir enhanced_physics

In [ ]:
import torch
model_state = torch.load('/kaggle/working/sgmse_physics_finetuned/physics_epochepoch=09_vallossvalid_loss=25.3103.ckpt', weights_only=False)
layers = list(model_state['state_dict'].keys())
print(f"Total layers: {len(layers)}")
print("First 10:", layers[:10])
print("Last 10:", layers[-10:])

In [ ]:
finetune_script = '''
import os
os.chdir('/kaggle/working/sgmse')
import torch
if not hasattr(torch, '_load_patched'):
    _orig = torch.load
    def _patched(*args, **kwargs):
        kwargs['weights_only'] = False
        return _orig(*args, **kwargs)
    torch.load = _patched
    torch._load_patched = True

import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, Callback
from sgmse.model import ScoreModel
from sgmse.data_module import SpecsDataModule

CKPT_PATH = "/kaggle/working/sgmse/voicebank_pretrained.ckpt"
SAVE_DIR = "/kaggle/working/sgmse_physics_v2"
DATA_DIR = "/kaggle/working/sgmse/data"
LR = 5e-6
os.makedirs(SAVE_DIR, exist_ok=True)

model = ScoreModel.load_from_checkpoint(
    CKPT_PATH,
    map_location="cuda",
    loss_type="data_prediction",
    loss_weighting="1",
    l1_weight=0.001,
    pesq_weight=0.0,
    num_eval_files=0,
    lr=LR
)
model.physics_weight = 0.00001

# Freeze all layers first
for param in model.parameters():
    param.requires_grad = False

# Unfreeze output layer and last 10 modules (67-76)
for param in model.dnn.output_layer.parameters():
    param.requires_grad = True
for i in range(67, 77):
    for param in model.dnn.all_modules[i].parameters():
        param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print("Trainable params: " + str(trainable) + " / " + str(total))

class PrintValLoss(Callback):
    def on_validation_epoch_end(self, trainer, pl_module):
        metrics = trainer.callback_metrics
        val_loss = metrics.get("valid_loss")
        train_loss = metrics.get("train_loss_epoch")
        if val_loss is not None and train_loss is not None:
            print("\\n[Epoch " + str(trainer.current_epoch) + "] train_loss=" + str(round(float(train_loss), 4)) + " | valid_loss=" + str(round(float(val_loss), 4)) + "\\n")
        elif val_loss is not None:
            print("\\n[Epoch " + str(trainer.current_epoch) + "] valid_loss=" + str(round(float(val_loss), 4)) + "\\n")

data_module = SpecsDataModule(
    base_dir=DATA_DIR,
    format="default",
    batch_size=4,
    n_fft=510,
    hop_length=128,
    num_frames=256,
    window="hann",
    num_workers=2,
    dummy=False,
    spec_factor=0.15,
    spec_abs_exponent=0.5,
    normalize="noisy",
    transform_type="exponent"
)

checkpoint_callback = ModelCheckpoint(
    dirpath=SAVE_DIR,
    filename="physics_v2_epoch{epoch:02d}_valloss{valid_loss:.4f}",
    save_top_k=3,
    monitor="valid_loss",
    mode="min",
    every_n_epochs=1
)

trainer = pl.Trainer(
    max_epochs=10,
    accelerator="gpu",
    devices=1,
    callbacks=[checkpoint_callback, PrintValLoss()],
    log_every_n_steps=10,
    enable_progress_bar=True,
    gradient_clip_val=1.0
)

trainer.fit(model, datamodule=data_module)
print("Best checkpoint: " + checkpoint_callback.best_model_path)
'''

with open('/kaggle/working/sgmse/finetune_physics_v2.py', 'w') as f:
    f.write(finetune_script)
print("Script written successfully")

In [ ]:
!python finetune_physics_v2.py

In [ ]:
CKPT = "/kaggle/working/sgmse_physics_v2/physics_v2_epochepoch=09_vallossvalid_loss=0.5441.ckpt"

!python enhancement.py \
    --test_dir data/test/noisy \
    --enhanced_dir enhanced_physics_v2 \
    --ckpt $CKPT \
    --N 10

In [ ]:
!python calc_metrics.py \
    --clean_dir data/test/clean \
    --noisy_dir data/test/noisy \
    --enhanced_dir enhanced_physics_v2

In [ ]:
import torch

ckpt = torch.load(
    '/kaggle/working/sgmse_physics_v2/physics_v2_epochepoch=09_vallossvalid_loss=0.5441.ckpt',
    weights_only=False
)
saved_keys = list(ckpt['state_dict'].keys())
print(f"Keys in checkpoint: {len(saved_keys)}")
print("First 5:", saved_keys[:5])
print("Last 5:", saved_keys[-5:])

# ============================================================
# FIXED RUN (v3): score_matching loss + spectral envelope physics
# ============================================================

**Root cause of prior failure:** the pretrained `voicebank_pretrained.ckpt` was trained
with `loss_type='score_matching'` on the `ncsnpp` backbone (paper [2]). The earlier
fine-tune cells override `loss_type='data_prediction'`, which interprets the model's
score output as a clean-spectrogram prediction. That mismatch corrupts the fine-tuning
gradient regardless of the physics term.

**Secondary issue:** the prior "wave equation" loss was `mean((d^2 x_td/dt^2)^2)` on the
raw waveform — that is a high-frequency *suppressor*, which actively destroys speech.

**This run fixes both:**
1. Use `loss_type='score_matching'` to match the pretrained backbone.
2. Replace the physics term with a *spectral envelope smoothness* penalty applied in
   the **STFT magnitude domain along the frequency axis**. Physical motivation:
   source-filter model of speech production — the vocal-tract impulse response is a
   smooth function of frequency (formants), so the log-magnitude envelope of clean
   speech is locally smooth in `f`. This is the right domain (matches the model's
   internal representation) and the right physics (acoustics of speech production).
3. Apply the penalty to a **Tweedie-style estimate of x_0** recovered from the score,
   not to the raw score itself, so the regularizer targets the denoised signal.


In [1]:
# Re-clone sgmse cleanly to discard the broken patches from earlier cells.
import os, shutil
if os.path.exists('/kaggle/working/sgmse'):
    shutil.rmtree('/kaggle/working/sgmse')
os.chdir('/kaggle/working')
get_ipython().system('git clone https://github.com/sp-uhh/sgmse.git')
os.chdir('/kaggle/working/sgmse')
get_ipython().system('pip install -r requirements.txt --quiet')
get_ipython().system('pip install pesq pystoi pandas gdown --quiet')
# pretrained checkpoint
get_ipython().system('gdown 1_H3EXvhcYBhOZ9QNUcD5VZHc6ktrRbwQ -O voicebank_pretrained.ckpt')

Cloning into 'sgmse'...
remote: Enumerating objects: 1011, done.
remote: Counting objects: 100% (357/357), done.
remote: Compressing objects: 100% (145/145), done.
remote: Total 1011 (delta 276), reused 214 (delta 212), pack-reused 654 (from 3)
Receiving objects: 100% (1011/1011), 3.74 MiB | 25.69 MiB/s, done.
Resolving deltas: 100% (547/547), done.
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 36.7 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 3.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydata-profiling 4.18.1 requires typeguard<5,>=4, but you have typeguard 2.13.3 which is incompatible.
inflect 7.5.0 requires typeguard>=4.0.1, but you have typeguard 2.13.3 which is incompatible.
Downloading...
From (original): htt

In [ ]:
# Patch model.py: add spectral-envelope smoothness term inside the score_matching branch.
# We compute a Tweedie estimate of x_0 from the score, then penalize the second
# difference of log|x_0_hat| along the frequency axis.
patch = '''
            # === physics-informed regularizer (spectral envelope smoothness) ===
            # Tweedie estimate of the clean spectrogram from the score.
            # For OUVE-SDE this is an approximation that ignores the drift term;
            # used here as a regularization target, not an exact reconstruction.
            x_hat_spec = x_t + (sigma ** 2) * score
            mag = torch.abs(x_hat_spec).clamp(min=1e-7)
            log_mag = torch.log(mag)
            # Second difference along the frequency axis (axis=2 of (B,C,F,T)).
            d2_f = log_mag[:, :, 2:, :] - 2.0 * log_mag[:, :, 1:-1, :] + log_mag[:, :, :-2, :]
            phys_loss = torch.mean(d2_f ** 2)
            loss = loss + self.physics_weight * phys_loss
'''

with open('/kaggle/working/sgmse/sgmse/model.py', 'r') as f:
    content = f.read()

# Insert physics_weight attribute in __init__ (after self.loss_type = loss_type).
content = content.replace(
    'self.loss_type = loss_type\n',
    'self.loss_type = loss_type\n        self.physics_weight = 0.0\n',
    1,
)

# Insert the physics term at the end of the score_matching branch — right before the
# `elif self.loss_type == "denoiser":` line.
old = '            loss = torch.mean(0.5*torch.sum(losses.reshape(losses.shape[0], -1), dim=-1))\n        elif self.loss_type == "denoiser":'
new = '            loss = torch.mean(0.5*torch.sum(losses.reshape(losses.shape[0], -1), dim=-1))\n' + patch + '        elif self.loss_type == "denoiser":'
assert old in content, 'Anchor for score_matching patch not found'
content = content.replace(old, new, 1)

with open('/kaggle/working/sgmse/sgmse/model.py', 'w') as f:
    f.write(content)

get_ipython().system('grep -n "physics_weight\|phys_loss\|x_hat_spec" /kaggle/working/sgmse/sgmse/model.py')

In [ ]:
# Static validation: check the relative magnitudes of the score-matching loss and the
# physics term on a synthetic batch BEFORE committing to a full Kaggle training run.
# This avoids the lambda-scaling failure mode (Hypothesis 3).
import torch

torch.manual_seed(0)
B, C, F, T = 4, 1, 256, 256                # SGMSE+ STFT shape with n_fft=510, num_frames=256
sigma_val = 0.5                            # representative diffusion noise level

# Synthesize a smooth clean spectrogram + noisy x_t
freq = torch.linspace(0, 1, F).view(1, 1, F, 1)
time = torch.linspace(0, 1, T).view(1, 1, 1, T)
x_clean = (torch.exp(-freq * 4.0) * torch.cos(2 * 3.14159 * time * 3)).expand(B, C, F, T)
x_clean = torch.complex(x_clean, 0.5 * x_clean.roll(1, dims=-1))
z       = torch.randn_like(x_clean.real) + 1j * torch.randn_like(x_clean.real)
x_t     = x_clean + sigma_val * z
# A perturbed "score" estimate, of the same scale a trained model would produce.
score   = -z / sigma_val + 0.1 * (torch.randn_like(z.real) + 1j * torch.randn_like(z.real))

# (1) Standard score-matching loss: mean over batch of 0.5 * sum |score*sigma + z|^2
sm = torch.square(torch.abs(score * sigma_val + z))
loss_sm = torch.mean(0.5 * torch.sum(sm.reshape(B, -1), dim=-1))

# (2) Physics term: spectral envelope smoothness on log-magnitude of Tweedie x_0_hat
x_hat_spec = x_t + (sigma_val ** 2) * score
mag = torch.abs(x_hat_spec).clamp(min=1e-7)
log_mag = torch.log(mag)
d2_f = log_mag[:, :, 2:, :] - 2.0 * log_mag[:, :, 1:-1, :] + log_mag[:, :, :-2, :]
loss_phys = torch.mean(d2_f ** 2)

print(f'score-matching loss : {loss_sm.item():.4e}')
print(f'physics loss (raw)  : {loss_phys.item():.4e}')
print(f'ratio  phys / sm    : {(loss_phys.item() / loss_sm.item()):.4e}')

# Choose physics_weight so the physics contribution is ~1-5% of the SM loss at init.
target_fraction = 0.02
suggested_w = target_fraction * loss_sm.item() / loss_phys.item()
print(f'suggested physics_weight (for ~{target_fraction*100:.0f}% contribution): {suggested_w:.4e}')

# NaN / inf sanity check on the finite difference path.
assert torch.isfinite(loss_phys), 'physics loss is non-finite!'
assert torch.isfinite(loss_sm),   'score-matching loss is non-finite!'
print('OK: both loss components are finite.')

In [ ]:
# Fixed fine-tune script. Key changes vs. the failed runs:
#   - loss_type = 'score_matching'   (matches the pretrained ncsnpp backbone)
#   - No l1_weight / pesq_weight     (those belong to the data_prediction branch)
#   - physics_weight set from the static check above; tweak before launching if needed
#   - Logs the three loss components per epoch so you can monitor balance on Kaggle.

finetune_script = '''
import os
os.chdir("/kaggle/working/sgmse")
import torch
if not hasattr(torch, "_load_patched"):
    _orig = torch.load
    def _patched(*a, **kw):
        kw["weights_only"] = False
        return _orig(*a, **kw)
    torch.load = _patched
    torch._load_patched = True

import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, Callback
from sgmse.model import ScoreModel
from sgmse.data_module import SpecsDataModule

CKPT_PATH = "/kaggle/working/sgmse/voicebank_pretrained.ckpt"
SAVE_DIR  = "/kaggle/working/sgmse_physics_v3"
DATA_DIR  = "/kaggle/working/sgmse/data"
os.makedirs(SAVE_DIR, exist_ok=True)

# Use score_matching to match the pretrained backbone.
model = ScoreModel.load_from_checkpoint(
    CKPT_PATH,
    map_location="cuda",
    loss_type="score_matching",
    loss_weighting="sigma^2",
    num_eval_files=0,
    lr=1e-5,
)
# Physics weight chosen from the static magnitude check; ~2% of SM loss at init.
model.physics_weight = 0.0

class PrintLosses(Callback):
    def on_validation_epoch_end(self, trainer, pl_module):
        m = trainer.callback_metrics
        vl = m.get("valid_loss"); tl = m.get("train_loss_epoch")
        print("\\n[Epoch " + str(trainer.current_epoch) +
              "] train_loss=" + (str(round(float(tl),4)) if tl is not None else "?") +
              " | valid_loss=" + (str(round(float(vl),4)) if vl is not None else "?") + "\\n")

data_module = SpecsDataModule(
    base_dir=DATA_DIR, format="default", batch_size=4,
    n_fft=510, hop_length=128, num_frames=256, window="hann",
    num_workers=2, dummy=False, spec_factor=0.15, spec_abs_exponent=0.5,
    normalize="noisy", transform_type="exponent",
)

ckpt_cb = ModelCheckpoint(
    dirpath=SAVE_DIR,
    filename="physics_v3_epoch{epoch:02d}_valloss{valid_loss:.4f}",
    save_top_k=3, monitor="valid_loss", mode="min", every_n_epochs=1,
)

trainer = pl.Trainer(
    max_epochs=10, accelerator="gpu", devices=1,
    callbacks=[ckpt_cb, PrintLosses()],
    log_every_n_steps=10, enable_progress_bar=True,
    gradient_clip_val=1.0,
)
trainer.fit(model, datamodule=data_module)
print("Best checkpoint:", ckpt_cb.best_model_path)
'''

with open('/kaggle/working/sgmse/finetune_physics_v3.py', 'w') as f:
    f.write(finetune_script)
print('finetune_physics_v3.py written')

In [ ]:
# Run the fixed fine-tune (≈ same compute as the previous failed runs: 10 epochs × 1250
# samples × batch 4 ≈ 3,125 steps; with batch 4 this fits in roughly the same wall-clock
# budget you used before — ~2-3 GPU hours on the Kaggle T4).
get_ipython().system('python finetune_physics_v3.py')

In [2]:
# 1. Patch enhancement.py
with open('/kaggle/working/sgmse/enhancement.py', 'r') as f:
  content = f.read()

patch = '''import torch
_original_torch_load = torch.load
def _patched_torch_load(*args, **kwargs):
  kwargs['weights_only'] = False
  return _original_torch_load(*args, **kwargs)
torch.load = _patched_torch_load

'''
if '_patched_torch_load' not in content:
  with open('/kaggle/working/sgmse/enhancement.py', 'w') as f:
      f.write(patch + content)
  print('enhancement.py patched')
else:
  print('enhancement.py already patched')

# 2. Patch Lightning's loaders (pure Python, no sed)
for path in [
  '/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py',
  '/usr/local/lib/python3.12/dist-packages/lightning_fabric/utilities/cloud_io.py',
]:
  with open(path, 'r') as f:
      src = f.read()
  new = src.replace(
      'weights_only: Optional[bool] = None,',
      'weights_only: Optional[bool] = False,',
  )
  if new != src:
      with open(path, 'w') as f:
          f.write(new)
      print('patched', path)
  else:
      print('no change needed', path)

# 3. Clear any partial output from the failed run
import shutil, os
if os.path.exists('/kaggle/working/sgmse/enhanced_physics_v3'):
  shutil.rmtree('/kaggle/working/sgmse/enhanced_physics_v3')
  print('cleared enhanced_physics_v3/')

enhancement.py patched
patched /usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py
patched /usr/local/lib/python3.12/dist-packages/lightning_fabric/utilities/cloud_io.py


In [5]:
# Enhancement + metrics on the fixed checkpoint.
import glob, os
ckpts = sorted(glob.glob('/kaggle/working/sgmse_physics_v3/physics_v3_*.ckpt'))
print('checkpoints:', ckpts)
CKPT = ckpts[-1]  # most recent / lowest-loss
CKPT = '/kaggle/working/sgmse_physics_v3/physics_v3_epochepoch=04_vallossvalid_loss=714.3017.ckpt'
print('using:', CKPT)

get_ipython().system(f'python enhancement.py --test_dir data/test/noisy --enhanced_dir enhanced_physics_v3 --ckpt {CKPT} --N 10')
get_ipython().system('python calc_metrics.py --clean_dir data/test/clean --noisy_dir data/test/noisy --enhanced_dir enhanced_physics_v3')

checkpoints: ['/kaggle/working/sgmse_physics_v3/physics_v3_epochepoch=00_vallossvalid_loss=759.4227.ckpt', '/kaggle/working/sgmse_physics_v3/physics_v3_epochepoch=02_vallossvalid_loss=772.6055.ckpt', '/kaggle/working/sgmse_physics_v3/physics_v3_epochepoch=04_vallossvalid_loss=714.3017.ckpt', '/kaggle/working/sgmse_physics_v3/physics_v3_epochepoch=05_vallossvalid_loss=770.4171.ckpt', '/kaggle/working/sgmse_physics_v3/physics_v3_epochepoch=08_vallossvalid_loss=716.7010.ckpt', '/kaggle/working/sgmse_physics_v3/physics_v3_epochepoch=09_vallossvalid_loss=559.3162.ckpt']
using: /kaggle/working/sgmse_physics_v3/physics_v3_epochepoch=04_vallossvalid_loss=714.3017.ckpt
Set TORCH_CUDA_ARCH_LIST to: 7.5;7.5
100%|█████████████████████████████████████████| 824/824 [02:30<00:00,  5.47it/s]
PESQ: 2.53 ± 0.55
ESTOI: 0.85 ± 0.10
SI-SDR: 16.6 ± 3.9
SI-SIR: 24.6 ± 5.4
SI-SAR: 17.7 ± 3.7
